<div style="
    background: linear-gradient(90deg, #0f2027 0%, #2c5364 100%);
    padding: 32px 0 32px 0;
    border-radius: 15px;
    text-align: center;
    margin-bottom: 30px;
">
  <h1 style="color: #fff; letter-spacing: 2px; font-size: 2.7rem; margin: 0; font-weight: 800;">
    Module 1: Scaling Python for AI Workloads (Part 1)
  </h1>
</div>


This notebook provides a step-by-step introduction to Ray Tasks, the fundamental building block of Ray that enables distributed computing.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>Overview</li>
  <li>Creating Remote Functions</li>
  <li>Executing Remote Functions</li>
  <li>Getting Results</li>
  <li>Putting It All Together</li>
  <li>Object store</li>
  <li>Chaining Tasks and Passing Data</li>
  <li>Task retries</li>
  <li>Task Runtime Environments</li>
  <li>Resource allocation and management</li>
  <li>Nested Tasks</li>
  <li>Pipeline data processing and waiting for results</li>
  <li>Ray Actors</li>
</ol>
</div>

## Imports

In [ ]:
import os
import random
import sys
import time

import numpy as np
import ray

## 1. Overview

### Ray Core at a glance

- **Scales your code** across many CPU cores, machines, and accelerators.  
- **Schedules arbitrary task graphs** thanks to its distributed scheduler.
- **Hides distributed-system overhead** with built-ins for  
  - fast data serialization and transfer,  
  - smart task placement, 
  - distributed memory & reference counting.

Ray’s higher-level libraries build on Ray Core to offer ready-made APIs for common workloads.


## 2. Creating Remote Functions

The first step in using Ray is to create remote functions. A remote function is a regular Python function that can be executed on any process in your cluster.

Given a simple Python function:

In [ ]:
def add(a, b):
    return a + b

add

Decorate the function with `@ray.remote` to turn it into a remote function.

In [ ]:
@ray.remote
def remote_add(a, b):
    return a + b

remote_add

## 3. Executing Remote Functions

Native python functions are invoked by calling them

In [ ]:
add(1, 2)

Remote ray functions are executed as tasks by calling them with `.remote()` suffix

In [ ]:
remote_add.remote(1, 2)

<div class="alert alert-info">
  <strong><a href="https://docs.ray.io/en/latest/ray-core/key-concepts.html#tasks" target="_blank">Tasks</a></strong> is a remote, stateless Python function invokation.
</div>


Here is what happens when you call `{remote_function}.remote`:
1. Ray schedules the function execution as a task in a separate process in the cluster
2. Ray returns an `ObjectRef` (a reference to the future result) to you **immediately** 
3. The cluster executes the actual computation in the background


In [ ]:
ref = remote_add.remote(1, 2)
ref

Here is a map of how Python code is translated into Ray tasks.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-core/python_to_ray_task_map_v2.png" alt="Python to Ray Task Map" width="800">

## 4. Getting Results

If we want to wait (block) and retrieve the corresponding object, we can use `ray.get`

In [ ]:
ray.get(ref)

## 5. Putting It All Together

Here are the three steps:
1. Create the remote function
2. Execute it remotely
3. Get the result when needed


<div class="alert alert-block alert-info">
    
__Activity: define and invoke a Ray task__

Define a remote function `sqrt_add` that accepts two arguments and performs the following steps:
1. computes the square-root of the first
2. adds the second
3. returns the result

Execute it with 2 different sets of parameters and collect the results

```python
# Hint: define the below as a remote function
def sqrt_add(a, b):
    ... 

# Hint: invoke it as a remote task and collect the results
```


</div>

In [ ]:
# Write your solution here

<div class="alert alert-block alert-info">

<details>

<summary> Click to see solution </summary>

```python
import math

@ray.remote
def sqrt_add(a, b):
    return math.sqrt(a) + b

ray.get([sqrt_add.remote(2, 3), sqrt_add.remote(5, 4)])
```

</details>

</div>


### 5.1. Note about Ray ID Specification

IDs for tasks and objects are build according to the [ID specification in Ray](https://github.com/ray-project/ray/blob/master/src/ray/design_docs/id_specification.md).

### 5.2. Anti-pattern: Calling ray.get in a loop harms parallelism

|<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/ray-core/ray-get-in-a-loop.png" width="700px" loading="lazy">|
|:--|
|ray.get() is a blocking call. Avoid calling it on every item (left panel). Calling only on the final result improves performance (right panel).|

When trying to collect results for multiple remote function invocations (tasks), don't block and wait for each one individually. Let's consider this remote function:

In [ ]:
@ray.remote
def expensive_square(x):
    time.sleep(5)
    return x**2

This implementation will block for each item in the loop:

In [ ]:
results = []
for item in range(4):
    output = ray.get(expensive_square.remote(item))
    results.append(output)
results

Schedule all remote calls, which are then processed in parallel. After scheduling the work, we can then request all the results at once.

In [ ]:
refs = []
for j in range(4):
    refs.append(expensive_square.remote(j))
results = ray.get(refs)
results

<div class="alert alert-info">
Read more about this <strong><a href="https://docs.ray.io/en/latest/ray-core/patterns/ray-get-loop.html" target="_blank">anti-pattern</a></strong>.
</div>

## 6. Object store

Each worker node has its own object store, and collectively, these form a shared object store across the cluster.

Remote objects are immutable. That is, their values cannot be changed after creation. This allows remote objects to be replicated in multiple object stores without needing to synchronize the copies.

|<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/ray-core/ray-cluster.png" width="700px" loading="lazy">|
|:--|
|A Ray cluster with a head node and two worker nodes. Highlighted in orange is distributed object store.|

<div class="alert alert-info">
  <strong><a href="https://docs.ray.io/en/latest/ray-core/key-concepts.html#objects" target="_blank">Object</a></strong> - tasks and actors create and work with remote objects, which can be stored anywhere in a cluster. These objects are accessed using <strong>ObjectRef</strong> and are cached in a distributed shared-memory <strong>object store</strong>.
</div>

Let's consider following example:

In [ ]:
large_matrix = np.random.rand(1024, 1024, 1024//8) # approx. 1 GB
size_in_bytes = sys.getsizeof(large_matrix)

print(f"large_matrix has: {size_in_bytes/1024/1024/1024:.2f} GB")

Add an object to the object store using `ray.put()`

In [ ]:
obj_ref = ray.put(large_matrix)
obj_ref

Use the `ray.get()` method to fetch the result of a remote object from an object ref

In [ ]:
large_mat_from_object_store = ray.get(obj_ref)

While the contents are the same, the object store contains a copy of the array which is not the same memory location. 

In [ ]:
np.array_equal(large_mat_from_object_store, large_matrix)

Array in store is in shared memory whereas array in notebook is local to the notebook process.

In [ ]:
large_mat_from_object_store is large_matrix

### 6.1 Pattern: Pass large objects **by reference**

Ray distinguishes between a **value** and an **Object Ref** when you pass arguments to remote functions:

* **Value** → Ray serializes the data and writes a fresh copy to the object store for every call.
* **Object Ref** → Ray forwards only a lightweight ID; the worker fetches the data once and reuses it.

Upload big, read‑only objects once, then circulate their `ObjectRef` to every consumer task. This avoids repeated serialization, network traffic, and object‑store pressure.

Pass by value only for small literals (< 100 KiB); otherwise, pass by reference.


In [ ]:
large_ref = ray.put(np.random.rand(1000, 1000))  # Approx. 8 MB -> place in object store


@ray.remote
def compute_sum(x):
    return int(x.sum())


# all tasks use same ObjectRef minimizing copies (memory efficient)
results = ray.get([compute_sum.remote(large_ref) for _ in range(100)])

## 7. Chaining Tasks and Passing Data

Let's say we now want to execute a graph of two tasks:
1. Square a value using `expensive_square`
2. Add 1 to the `expensive_square` result, by using `remote_add`

This can be achieved without fetching an intermediate result.

Anti-pattern:

In [ ]:
# 1st task
square_ref = expensive_square.remote(2)
square_value = ray.get(square_ref)  # wait to get the value

# 2nd task
sum_ref = remote_add.remote(1, square_value)  # pass value from 1st task
sum_value = ray.get(sum_ref)

Instead, chain the tasks by passing the `ObjectRef` directly to the second task:

In [ ]:
square_ref = expensive_square.remote(2)
sum_ref = remote_add.remote(1, square_ref)
sum_value = ray.get(sum_ref)

## 8. Task retries

Let's consider two types of exceptions:
1. **system errors** (e.g., Python-level exceptions)
2. **application-level errors** (e.g., a machine fails)

Ray will automatically **retry a task up to 3 times**, if it fails due to a system error (e.g., a worker node dies).

Below task won't be retried by default because it's an application failure

In [ ]:
@ray.remote
def incorrect_square(x: int, prob: float) -> int:
    # Simulate potential failures
    if random.random() < prob:  # % chance of failure
        raise ValueError("Random failure")
    return x**2

In [ ]:
try:
    ray.get([incorrect_square.remote(x=4, prob=0.5) for _ in range(10)])
except ray.exceptions.RayTaskError:
    print("At least one of the tasks failed", flush=True)

Ray let's you specify how to handle retries when an exception is encountered.

Let's retry on `ValueError`, like below:

In [ ]:
@ray.remote(retry_exceptions=[ValueError])
def correct_square(x: int, prob: float) -> int:
    # Simulate potential failures
    if random.random() < prob:  # % chance of failure
        raise ValueError("Random failure")

    return x**2

Note we did not have to re-define the remote function, instead we can an update version using `.options`

In [ ]:
correct_square_mod = correct_square.options(
    retry_exceptions=[ValueError], max_retries=10,
)

Let's try it out:

In [ ]:
try:
    outputs = ray.get([correct_square_mod.remote(x=4, prob=0.5) for _ in range(10)])
except ray.exceptions.RayTaskError:
    print("At least one of the tasks failed", flush=True)

outputs

<div class="alert alert-info">
Refer to the <strong><a href="https://docs.ray.io/en/latest/ray-core/tasks/retries.html" target="_blank">retries</a></strong> to learn more.
</div>

## 9. Task Runtime Environments

Runtime environments can be used on top of the prepared environment from the Ray Cluster to customize the execution of tasks.

When setting up a worker process to run a task, Ray will first prepare the environment for the task.

This includes things like:
* installing dependencies
* setting environment variables

For example, we can set an environment variable:

In [ ]:
@ray.remote(runtime_env={"env_vars": {"my_custom_env": "prod"}})
def f():
    env = os.environ["my_custom_env"]
    return f"My custom env is {env}"

In [ ]:
ray.get(f.remote())

## 10. Resource allocation and management

By default, Ray will schedule a task as long as there is at least one CPU available.

In code this can be specified in the `ray.remote`, like this:

In [ ]:
@ray.remote(num_cpus=1)
def remote_add(a, b):
    return a + b

However, these resource specifications are not enforced - i.e. they are entirely [logical and not physical](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html#physical-resources-vs-logical-resources).

This means that you can for instance perform multiprocessing ormultithreading within a task and oversubscribe to resources.

In [ ]:
@ray.remote(num_cpus=1)
def mm(n: int = 4000):
    A = np.random.rand(n, n)
    B = np.random.rand(n, n)

    # Time the dot product
    start = time.time()
    np.dot(A, B)
    end = time.time()
    print(f"Took {end - start}s")
    
ray.get(mm.options(runtime_env={"env_vars": {"OMP_NUM_THREADS": "1"}}).remote())
ray.get(mm.options(runtime_env={"env_vars": {"OMP_NUM_THREADS": "8"}}).remote())

<div class="alert alert-info">

Note by default, Ray will set the `OMP_NUM_THREADS` environment variable to the number of CPUs in the cluster.

Learn more about <strong><a href="https://docs.ray.io/en/latest/ray-core/scheduling/resources.html#physical-resources-and-logical-resources" target="_blank">physical resources and logical resources</a></strong>.
</div>

### 10.1. Note on resources requests, available resources, configuring large clusters

<p>During the <em>scheduling stage</em>, Ray evaluates the <strong>resource requirements</strong> specified via the <code>@ray.remote</code> decorator or within the <code>resources={...}</code> argument. These requirements may include:</p>

<ul>
    <li><strong>CPU</strong> e.g., <code>@ray.remote(num_cpus=2)</code>)</li>
    <li><strong>GPU</strong> e.g., <code>@ray.remote(num_gpus=1)</code>)</li>
    <li><strong>Custom resources</strong>: User-defined custom resources like <code>"TPU"</code></li>
    <li><strong>Memory</strong></li>
</ul>

<p>Ray's scheduler checks the <strong>resource specification</strong> (sometimes referred to as <strong>resource shape</strong>) to match tasks and actors with available resources in the cluster. If the exact resource combination is unavailable, Ray may autoscaler the cluster.</p>

<p>You can inspect the current resource availability using:</p>
<pre><code>
ray.available_resources()
</code></pre>

<p>This returns a dictionary showing the currently available CPUs, GPUs, memory, and any custom resources, for example:</p>

<pre><code>{'CPU': 24.0, 'GPU': 1.0, 'memory': 2147483648.0}</code></pre>

In [ ]:
ray.available_resources()

<div class="alert alert-info">

<strong>Pattern:</strong> configure the head node to be unavailable for compute tasks.

When scaling to large clusters, it's important to ensure that the <strong>head node</strong> does not handle any compute tasks. Users can indicate that the head node is unavailable for compute by setting its resources:

```resources: {"CPU": 0}```

Learn more about <strong><a href="https://docs.ray.io/en/latest/cluster/vms/user-guides/large-cluster-best-practices.html#configuring-the-head-node" target="_blank">configuring the head node</a></strong>.
</div>

### 10.2. Fractional resources

Fractional resources allow Ray Tasks to request a fraction of a CPU or GPU (e.g., 0.5), enabling finer-grained resource allocation.

Let's consider the above example again:

In [ ]:
@ray.remote(num_cpus=0.5)
def remote_add(a, b):
    return a + b

This means Ray will allow execution of 2x the number of CPUs on the machine to run the task.

In [ ]:
ref = remote_add.remote(2, 3)
ref

In [ ]:
ray.get(ref)

<div class="alert alert-info">
    Fractional resources include support for <strong><a href="https://docs.ray.io/en/latest/ray-core/scheduling/accelerators.html#fractional-accelerators" target="_blank">multiple accelerators</a></strong>, allowing users to load multiple smaller models onto a single GPU. This is especially useful for scenarios like model inference. Learn more about <strong><a href="https://docs.ray.io/en/latest/ray-core/scheduling/resources.html#fractional-resource-requirements" target="_blank">fractional resource requirements</a></strong>.
</div>

## 11. Nested Tasks

Ray __does not__ require that all of your tasks and their dependencies be submitted from one "driver" process.

For example, you can have a "main" task that submits other tasks and then waits for them to complete.

In [ ]:
@ray.remote
def main():
    square_ref_1 = expensive_square.remote(1)
    square_ref_2 = expensive_square.remote(2)
    add_ref = remote_add.remote(square_ref_1, square_ref_2)
    return ray.get(add_ref)

Let's run the main task to dynamically schedule the other tasks:

In [ ]:
ray.get(main.remote())

In this example:
1. Our local process requests Ray to schedule a `main` task in the cluster
2. Ray executes the `main` task in a separate worker process
3. Inside `main`, we invoke multiple `expensive_square` tasks, which Ray distributes across available worker processes
4. Once all "sub tasks" complete, `main` returns the final value of the `remote_add` task

This ability for tasks to schedule other tasks using uniform semantics makes Ray particularly powerful and flexible.

<div class="alert alert-info">
To avoid deadlocks, Ray yields CPU resources while blocked waiting for a task to complete. Read more about <strong><a href="https://docs.ray.io/en/latest/ray-core/tasks/nested-tasks.html#yielding-resources-while-blocked" target="_blank">yielding resources while blocked</a></strong>.
</div>

## 12. Pipeline data processing and waiting for results

After launching a number of tasks, you may want to know which ones have finished executing without blocking on all of them. This could be achieved by `ray.wait()`

|<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/ray-core/pipeline-data-processing.png" width="400px" loading="lazy">|
|:--|
|(top panel) Execution timeline when using ray.get() to wait for all results before calling process results. (bottom panel) Execution timeline when using ray.wait() to process results as soon as they become available.|

Here are functions to match the above diagram:

In [ ]:
@ray.remote
def do_some_work(x):
    time.sleep(random.uniform(0, 4))  # Replace this with work you need to do.
    return x


def process_incremental(sum, result):
    time.sleep(1)  # Replace this with some processing code.
    return sum + result


def process_results(results):
    sum = 0
    for x in results:
        sum += process_incremental(sum, x)
    return sum

This is the **naive approach:**, block until all tasks are complete and then process the results.

In [ ]:
start = time.time()
data_list = ray.get([do_some_work.remote(x) for x in range(20)])
sum = process_results(data_list)
print("duration =", time.time() - start, "\nresult = ", sum)

This is the **pipelined** approach, process items as soon as they become available

In [ ]:
start = time.time()
result_ids = [do_some_work.remote(x) for x in range(20)]

sum = 0
while len(result_ids):
    done_id, result_ids = ray.wait(result_ids)
    sum = process_incremental(sum, ray.get(done_id[0]))

print("duration =", time.time() - start, "\nresult = ", sum)

<div class="alert alert-info">
Read more about the <strong><a href="https://docs.ray.io/en/latest/ray-core/tips-for-first-time.html#tip-4-pipeline-data-processing" target="_blank">pipeline data processing</a></strong>
</div>

## 13. Ray Actors

Actors extend the Ray API from functions (tasks) to classes.

An actor is a stateful worker. When a new actor is instantiated, a new worker is created, and methods of the actor are scheduled on that specific worker and can access and mutate the state of that worker. Similarly to Ray Tasks, actors support CPU and GPU compute as well as fractional resources.

Let's look at an example of an actor which maintains a running balance.

In [ ]:
@ray.remote
class Accounting:
    def __init__(self):
        self.total = 0
    
    def add(self, amount):
        self.total += amount
        
    def remove(self, amount):
        self.total -= amount
        
    def total(self):
        return self.total

<div class="alert alert-info">
  <strong><a href="https://docs.ray.io/en/latest/ray-core/key-concepts.html#actors" target="_blank">Actor</a></strong> is a remote, stateful Python class.
</div>

<div class="alert alert-info">

The most common use case for actors is with state that is not mutated but is large enough that we may want to load it only once and ensure we can route calls to it over time, such as a large AI model.

</div>

Define an actor with the `@ray.remote` decorator and then use `<class_name>.remote()` ask Ray to construct and instance of this actor somewhere in the cluster.

We get an actor handle which we can use to communicate with that actor, pass to other code, tasks, or actors, etc.

In [ ]:
acc = Accounting.remote()

We can send a message to an actor -- with RPC semantics -- by using `<handle>.<method_name>.remote()`

In [ ]:
acc.total.remote()

Not surprisingly, we get an object ref back

In [ ]:
ray.get(acc.total.remote())

We can mutate the state inside this actor instance

In [ ]:
acc.add.remote(100)

In [ ]:
acc.remove.remote(10)

In [ ]:
ray.get(acc.total.remote())

<div class="alert alert-block alert-info">

__Activity: linear model inference__

* Create an actor which applies a model to convert Celsius temperatures to Fahrenheit
* The constructor should take model weights (w1 and w0) and store them as instance state
* A convert method should take a scalar, multiply it by w1 then add w0 (weights retrieved from instance state) and then return the result


```python
# Hint: define the below as a remote actor
class LinearModel:
    def __init__(self, w0, w1):
        # Hint: store the weights

    def convert(self, celsius):
        # Hint: convert the celsius temperature to Fahrenheit

# Hint: create an instance of the LinearModel actor

# Hint: convert 100 Celsius to Fahrenheit
```

</div>

In [ ]:
# Write your solution here

(raylet) WARNING: 4 PYTHON worker processes have been started on node: db597ec2c8fd404787b86e5e023626385348eebdd00c49f5b888d0f0 with address: 10.0.122.124. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).
(raylet) WARNING: 6 PYTHON worker processes have been started on node: db597ec2c8fd404787b86e5e023626385348eebdd00c49f5b888d0f0 with address: 10.0.122.124. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).
(raylet) WARNING: 8 PYTHON worker processes have been started on node: db597ec2c8fd404787b86e5e023626385348eebdd00c49f5b888d0f0 with address: 10.0.122.124. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644

<div class="alert alert-block alert-info">

<details>

<summary> Click to see solution </summary>

```python
@ray.remote
class LinearModel:
    def __init__(self, w0, w1):
        self.w0 = w0
        self.w1 = w1

    def convert(self, celsius):
        return self.w1 * celsius + self.w0

model = LinearModel.remote(w1=9/5, w0=32)
ray.get(model.convert.remote(100))
```

</details>
</div>


<!-- TODO: add Patterns/antipatterns based on above learnings-->
